<a href="https://colab.research.google.com/github/krimits/hotel-review-nlp/blob/main/notebooks/05_distilbert_full_and_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DistilBERT full fine-tuning vs from-scratch LoRA

This Colab notebook produces the missing, directly comparable encoder experiment for the project. It trains both models on the **same ordered legacy splits**:

| split | rows | negative | positive |
|---|---:|---:|---:|
| train | 118,990 | 26,232 | 92,758 |
| dev | 14,872 | 3,278 | 11,594 |
| test | 13,278 | 3,278 | 10,000 |

It refuses to run when a split differs by even one text, label, or row position. The test set is used only after checkpoint selection by dev macro-F1. Each run writes its merged model, tokenizer, metrics, logs, dev/test logits, and labels. The final cells create a small handoff ZIP and a full artifact ZIP.

The legacy splits contain known normalized text overlap across splits. This notebook preserves them only to make the new runs comparable with the historical 118,990-row baselines; it does not present them as a leakage-clean final dataset.

## 1. Clone the flattened repository and install the pinned stack

Select a GPU runtime first: **Runtime → Change runtime type → T4 GPU** (or a faster GPU). The clone must contain `pyproject.toml` directly at its root; the assertion below catches the old nested layout.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/krimits/hotel-review-nlp.git"
REPO_REF = "main"
REPO_DIR = Path("/content/hotel-review-nlp")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Reusing existing checkout:", REPO_DIR)
assert (REPO_DIR / "pyproject.toml").is_file(), "Flattened root pyproject.toml is missing"
assert not (REPO_DIR / "hotel-review-nlp" / "pyproject.toml").exists(), (
    "The remote still has the obsolete nested project layout. Push the flatten commit first."
)

pinned = [
    "transformers==4.56.2",
    "accelerate==1.10.1",
    "peft==0.17.1",
    "datasets==3.6.0",
    "pytest>=8,<9",
    "trackio==0.37.1",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pinned], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

GIT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print("Repository:", REPO_DIR)
print("Commit:", GIT_COMMIT)

### CUDA preflight

This is a hard gate. CPU execution is intentionally rejected because two full-data encoder runs are not practical there. Free Colab runtimes can disconnect; the current trainer saves the final best model only after training completes, so use Drive persistence and run one training cell at a time.

In [ ]:
import importlib.metadata as metadata
import json
import platform

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Attach a GPU runtime and rerun from the top.")

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print("GPU:", gpu_name)
print("Compute capability:", capability)
print(f"CUDA memory: {free_bytes / 2**30:.1f} GiB free / {total_bytes / 2**30:.1f} GiB total")
print("Torch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
subprocess.run(["nvidia-smi"], check=True)

## 2. Choose persistence and provide the frozen parquet files

`data/processed/*.parquet` is intentionally ignored by Git, so Colab cannot obtain these splits from the repository. Leave `DATA_SOURCE = "upload"` and upload either:

- `train.parquet`, `dev.parquet`, and `test.parquet`, or
- one ZIP containing those three filenames at any directory depth.

To prepare a ZIP locally from the flattened repository in PowerShell:

```powershell
Compress-Archive -LiteralPath data\processed\train.parquet,data\processed\dev.parquet,data\processed\test.parquet -DestinationPath legacy_frozen_splits.zip -Force
```

Alternatively, place the ZIP or a directory containing the three files in Drive and set `DATA_SOURCE = "drive"` plus `DRIVE_DATA_PATH`.

In [ ]:
DATA_SOURCE = "upload"  # @param ["upload", "drive"]
DRIVE_DATA_PATH = "/content/drive/MyDrive/hotel-review-nlp-data/legacy_frozen_splits.zip"  # @param {type:"string"}
USE_GOOGLE_DRIVE_FOR_OUTPUTS = True  # @param {type:"boolean"}
EXPERIMENT_ID = "distilbert_legacy_full_v1"  # @param {type:"string"}

from google.colab import drive, files

if DATA_SOURCE == "drive" or USE_GOOGLE_DRIVE_FOR_OUTPUTS:
    drive.mount("/content/drive")

SOURCE_DATA = REPO_DIR / "data" / "processed"
SOURCE_DATA.mkdir(parents=True, exist_ok=True)
if USE_GOOGLE_DRIVE_FOR_OUTPUTS:
    OUTPUT_ROOT = Path("/content/drive/MyDrive/hotel-review-nlp-results")
else:
    OUTPUT_ROOT = Path("/content/hotel-review-nlp-results")
EXPERIMENT_DIR = OUTPUT_ROOT / EXPERIMENT_ID
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print("Input staging directory:", SOURCE_DATA)
print("Persistent experiment directory:", EXPERIMENT_DIR)

### Stage inputs without unsafe ZIP extraction

ZIP members are read by basename and written only to the fixed staging directory. Existing staged inputs make this cell rerunnable; the following fingerprint gate still detects corrupt or incorrect files.

In [ ]:
import io
import zipfile

REQUIRED_FILES = {f"{split}.parquet" for split in ("train", "dev", "test")}

def write_zip_members(blob: bytes) -> None:
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        candidates = {}
        for info in archive.infolist():
            basename = Path(info.filename).name
            if basename in REQUIRED_FILES:
                if basename in candidates:
                    raise ValueError(f"ZIP contains duplicate {basename} entries")
                candidates[basename] = info
        missing = REQUIRED_FILES - set(candidates)
        if missing:
            raise FileNotFoundError(f"ZIP is missing: {sorted(missing)}")
        for basename, info in candidates.items():
            (SOURCE_DATA / basename).write_bytes(archive.read(info))

def copy_from_drive(source: Path) -> None:
    if source.is_dir():
        for basename in REQUIRED_FILES:
            candidate = source / basename
            if not candidate.is_file():
                raise FileNotFoundError(candidate)
            shutil.copy2(candidate, SOURCE_DATA / basename)
    elif source.is_file() and source.suffix.lower() == ".zip":
        write_zip_members(source.read_bytes())
    else:
        raise FileNotFoundError(f"Expected a ZIP or directory at {source}")

missing = {name for name in REQUIRED_FILES if not (SOURCE_DATA / name).is_file()}
if missing and DATA_SOURCE == "upload":
    print("Upload the three parquet files or their ZIP:", sorted(missing))
    uploaded = files.upload()
    for uploaded_name, blob in uploaded.items():
        basename = Path(uploaded_name).name
        if basename.lower().endswith(".zip"):
            write_zip_members(blob)
        elif basename in REQUIRED_FILES:
            (SOURCE_DATA / basename).write_bytes(blob)
elif missing and DATA_SOURCE == "drive":
    copy_from_drive(Path(DRIVE_DATA_PATH))

remaining = {name for name in REQUIRED_FILES if not (SOURCE_DATA / name).is_file()}
if remaining:
    raise FileNotFoundError(f"Frozen split files are still missing: {sorted(remaining)}")
print("Staged:", sorted(path.name for path in SOURCE_DATA.glob("*.parquet")))

### Exact byte and semantic fingerprint gate

Two checks are required. The parquet byte hashes prove that the uploaded files are the archived local files. The order-sensitive semantic hashes prove the exact `(text, label)` sequence consumed by training. No sampling or resplitting occurs.

In [ ]:
import hashlib
import pandas as pd

from reviewnlp.utils.experiments import fingerprint_splits

EXPECTED_FILE_SHA256 = {
    "train": "ce52f8f34003f23c161d0e0a001b1009e80ca094c85a3522232a815748860fac",
    "dev": "cb6e0cff8fb264cd0c21fd990278c0bb95106572baa1b5512cc23e011ac317d1",
    "test": "8b37675028c8081951ee0ebd1181aaf484d63058122b7088288266fe992a7be1",
}
EXPECTED_FINGERPRINTS = {
    "train": {
        "rows": 118_990,
        "negative": 26_232,
        "positive": 92_758,
        "sha256": "1a5111176aafc286a7c784c9dec3abc2528b4b94d67fc8e7000ba4df4bfbd627",
    },
    "dev": {
        "rows": 14_872,
        "negative": 3_278,
        "positive": 11_594,
        "sha256": "7c080936decc0b470cec4eb162c121fc1245643ba095fa4d602529bc9baa8e9f",
    },
    "test": {
        "rows": 13_278,
        "negative": 3_278,
        "positive": 10_000,
        "sha256": "a02c21271639640d645d729b959ce666671d4ba4ed5abfd43e6f9b5729245730",
    },
}

observed_file_sha256 = {
    split: hashlib.sha256((SOURCE_DATA / f"{split}.parquet").read_bytes()).hexdigest()
    for split in ("train", "dev", "test")
}
assert observed_file_sha256 == EXPECTED_FILE_SHA256, (
    "Parquet byte hashes differ. Upload the exact frozen legacy files.",
    observed_file_sha256,
)

observed_fingerprints = fingerprint_splits(SOURCE_DATA)
assert observed_fingerprints == EXPECTED_FINGERPRINTS, (
    "Ordered text/label fingerprints differ.",
    observed_fingerprints,
)
display(pd.DataFrame(observed_fingerprints).T)
print("Frozen data fingerprint gate: PASS")

### Record the legacy overlap limitation and environment

The overlap counts are asserted rather than hidden. They are part of this historical dataset's identity. A later leakage-clean benchmark must use separately versioned splits and must not be compared as if it were this experiment.

In [ ]:
SPLITS = ("train", "dev", "test")
frames = {
    split: pd.read_parquet(SOURCE_DATA / f"{split}.parquet")
    for split in SPLITS
}
normalized_texts = {
    split: set(frame["text"].astype(str).str.strip().str.casefold())
    for split, frame in frames.items()
}
observed_overlap = {
    "train-dev": len(normalized_texts["train"] & normalized_texts["dev"]),
    "train-test": len(normalized_texts["train"] & normalized_texts["test"]),
    "dev-test": len(normalized_texts["dev"] & normalized_texts["test"]),
}
EXPECTED_LEGACY_OVERLAP = {"train-dev": 180, "train-test": 170, "dev-test": 24}
assert observed_overlap == EXPECTED_LEGACY_OVERLAP, observed_overlap
print("Known normalized cross-split overlap:", observed_overlap)

source_hashes = {
    str(path.relative_to(REPO_DIR)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in sorted((REPO_DIR / "src").rglob("*.py"))
}
environment = {
    "git_commit": GIT_COMMIT,
    "repo_ref": REPO_REF,
    "python": sys.version,
    "platform": platform.platform(),
    "gpu": gpu_name,
    "compute_capability": list(capability),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "packages": {
        name: metadata.version(name)
        for name in [
            "transformers", "accelerate", "peft", "datasets",
            "pandas", "numpy", "scikit-learn", "pyarrow", "trackio",
        ]
    },
    "source_sha256": source_hashes,
}
data_manifest = {
    "purpose": "historical full-data comparison; known cross-split overlap retained",
    "parquet_file_sha256": observed_file_sha256,
    "semantic_fingerprints": observed_fingerprints,
    "normalized_cross_split_overlap": observed_overlap,
}

def write_or_verify_json(path: Path, value: dict) -> None:
    if path.exists():
        previous = json.loads(path.read_text(encoding="utf-8"))
        if previous != value:
            raise RuntimeError(f"{path.name} changed; choose a new EXPERIMENT_ID")
    else:
        path.write_text(json.dumps(value, indent=2), encoding="utf-8")

write_or_verify_json(EXPERIMENT_DIR / "environment.json", environment)
write_or_verify_json(EXPERIMENT_DIR / "data_manifest.json", data_manifest)
print("Environment and data manifests recorded.")

## 3. Verify the custom LoRA implementation

The repository's offline unit tests check adapter injection, frozen base parameters, trainable task head, gradients, merge behavior, and numerical agreement with PEFT on a tiny model. Training starts only after they pass.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_lora.py", "-v", "-o", "addopts="],
    cwd=REPO_DIR,
    check=True,
)

## 4. Lock the shared experiment configuration

Full FT and scratch LoRA share the base checkpoint, exact data, tokenizer, max length, seed, batch sizes, epochs, scheduler, weight decay, and dev-based checkpoint selection. The only intended optimization difference is learning rate: `2e-5` for full FT and `1e-4` for LoRA. LoRA trains rank-8 adapters on DistilBERT `q_lin`/`v_lin` plus the newly initialized classification head.

In [ ]:
import yaml

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
EPOCHS = 2
FULL_LR = 2e-5
LORA_LR = 1e-4
LORA_RANK = 8
LORA_ALPHA = 16.0
LORA_DROPOUT = 0.0

FULL_DIR = EXPERIMENT_DIR / "distilbert"
LORA_DIR = EXPERIMENT_DIR / "distilbert_lora_scratch"
encoder_config = {
    "seed": SEED,
    "model": {"name": MODEL_NAME, "max_length": MAX_LENGTH},
    "data": {"processed_dir": str(SOURCE_DATA), "train_cap": None},
    "train": {
        "batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": FULL_LR,
        "weight_decay": 0.01,
        "warmup_ratio": 0.06,
        "fp16": True,
    },
    "output": {"model_dir": str(FULL_DIR)},
}
print(yaml.safe_dump(encoder_config, sort_keys=False))

### Training helper

Each model runs in its own Python process, which releases its GPU memory before the next run. Logs stream to the notebook and are also saved. A completed run is reused only when its recorded data fingerprints and request match exactly.

In [ ]:
def validate_completed_run(output_dir: Path) -> dict:
    required = {
        "config.json", "metrics.json", "dev_logits.npy", "dev_labels.npy",
        "test_logits.npy", "test_labels.npy", "tokenizer_config.json",
    }
    missing = sorted(name for name in required if not (output_dir / name).is_file())
    if missing:
        raise FileNotFoundError(f"Incomplete run {output_dir}: {missing}")
    metrics = json.loads((output_dir / "metrics.json").read_text(encoding="utf-8"))
    if metrics["data"] != EXPECTED_FINGERPRINTS:
        raise RuntimeError(f"{output_dir} was evaluated on different data")
    return metrics

def run_training(module: str, output_dir: Path, extra_args: list[str] | None = None) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)
    config_path = output_dir / "run_config.yaml"
    request = {
        "module": module,
        "git_commit": GIT_COMMIT,
        "config": encoder_config,
        "extra_args": extra_args or [],
    }
    request_path = output_dir / "request.json"
    if request_path.exists():
        previous = json.loads(request_path.read_text(encoding="utf-8"))
        if previous != request:
            raise RuntimeError(f"Settings changed for {output_dir}; use a new EXPERIMENT_ID")
    else:
        request_path.write_text(json.dumps(request, indent=2), encoding="utf-8")
    config_path.write_text(yaml.safe_dump(encoder_config, sort_keys=False), encoding="utf-8")

    metrics_path = output_dir / "metrics.json"
    if metrics_path.exists():
        print("Reusing validated completed run:", output_dir)
        return validate_completed_run(output_dir)

    command = [
        sys.executable, "-u", "-m", module, "--config", str(config_path),
        *(extra_args or []),
    ]
    print("Running:", " ".join(command))
    log_path = output_dir / "train.log"
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            return_code = process.wait()
    if return_code:
        raise RuntimeError(f"Training failed with exit code {return_code}; see {log_path}")
    return validate_completed_run(output_dir)

## 5. Run full DistilBERT fine-tuning

This is the missing 118,990-row full fine-tune. Let the cell finish and confirm that `metrics.json` appears in Drive before starting LoRA.

In [ ]:
full_metrics = run_training(
    "reviewnlp.llm.train_distilbert",
    FULL_DIR,
)
display(pd.DataFrame([{
    "model": "DistilBERT full FT",
    "best_dev_macro_f1": full_metrics["best_dev_macro_f1"],
    "test_macro_f1": full_metrics["test"]["macro_f1"],
    "test_accuracy": full_metrics["test"]["accuracy"],
    "trainable_params": full_metrics["trainable_params"],
    "training_minutes": full_metrics["training_seconds"] / 60,
    "peak_cuda_mb": full_metrics["peak_cuda_memory_mb"],
}]).round(4))

## 6. Run DistilBERT with from-scratch LoRA

The custom adapters and task head are trained; the encoder base stays frozen. At the end, the repository saves both `lora_scratch.pt` and a merged Hugging Face checkpoint for ordinary inference.

In [ ]:
lora_metrics = run_training(
    "reviewnlp.llm.train_distilbert_lora",
    LORA_DIR,
    [
        "--r", str(LORA_RANK),
        "--alpha", str(LORA_ALPHA),
        "--dropout", str(LORA_DROPOUT),
        "--lr", str(LORA_LR),
    ],
)
display(pd.DataFrame([{
    "model": "DistilBERT scratch LoRA + head",
    "best_dev_macro_f1": lora_metrics["best_dev_macro_f1"],
    "test_macro_f1": lora_metrics["test"]["macro_f1"],
    "test_accuracy": lora_metrics["test"]["accuracy"],
    "trainable_params": lora_metrics["trainable_params"],
    "training_minutes": lora_metrics["training_seconds"] / 60,
    "peak_cuda_mb": lora_metrics["peak_cuda_memory_mb"],
}]).round(4))

## 7. Verify comparability, calculate McNemar, and reload checkpoints

This gate requires identical saved labels and exact input fingerprints. It computes an exact paired McNemar test from the two saved prediction arrays and verifies that each merged checkpoint reproduces its cached predictions on the first 32 test rows.

In [ ]:
import gc
import numpy as np

from reviewnlp.evaluation.significance import pairwise_mcnemar
from reviewnlp.llm.predict import predict_encoder

full_metrics = validate_completed_run(FULL_DIR)
lora_metrics = validate_completed_run(LORA_DIR)
assert full_metrics["data"] == lora_metrics["data"] == EXPECTED_FINGERPRINTS

full_labels = np.load(FULL_DIR / "test_labels.npy")
lora_labels = np.load(LORA_DIR / "test_labels.npy")
np.testing.assert_array_equal(full_labels, lora_labels)
assert len(full_labels) == EXPECTED_FINGERPRINTS["test"]["rows"]

full_predictions = np.load(FULL_DIR / "test_logits.npy").argmax(axis=-1)
lora_predictions = np.load(LORA_DIR / "test_logits.npy").argmax(axis=-1)
mcnemar = pairwise_mcnemar(
    full_labels,
    {"DistilBERT full FT": full_predictions, "DistilBERT scratch LoRA": lora_predictions},
)

comparison_rows = []
for model_name, metrics in [
    ("DistilBERT full FT", full_metrics),
    ("DistilBERT scratch LoRA + head", lora_metrics),
]:
    comparison_rows.append({
        "model": model_name,
        "git_commit": GIT_COMMIT,
        "train_rows": EXPECTED_FINGERPRINTS["train"]["rows"],
        "trainable_params": metrics["trainable_params"],
        "adapter_params": metrics["adapter_params"],
        "head_params": metrics["head_params"],
        "total_params": metrics["total_params"],
        "trainable_pct": 100 * metrics["trainable_params"] / metrics["total_params"],
        "best_dev_macro_f1": metrics["best_dev_macro_f1"],
        "test_macro_f1": metrics["test"]["macro_f1"],
        "test_accuracy": metrics["test"]["accuracy"],
        "training_minutes": metrics["training_seconds"] / 60,
        "peak_cuda_mb": metrics["peak_cuda_memory_mb"],
        "learning_rate": metrics["settings"]["lr"],
    })
comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))
print(json.dumps(mcnemar, indent=2))

comparison.to_csv(EXPERIMENT_DIR / "distilbert_comparison.csv", index=False)
(EXPERIMENT_DIR / "distilbert_comparison.json").write_text(
    json.dumps(comparison_rows, indent=2), encoding="utf-8"
)
(EXPERIMENT_DIR / "mcnemar_full_vs_lora.json").write_text(
    json.dumps(mcnemar, indent=2), encoding="utf-8"
)

sample = frames["test"].head(32)
label_names = np.array(["negative", "positive"])
for name, model_dir, cached_predictions in [
    ("full FT", FULL_DIR, full_predictions),
    ("scratch LoRA", LORA_DIR, lora_predictions),
]:
    reloaded = predict_encoder(
        str(model_dir), sample["text"].astype(str).tolist(),
        max_length=MAX_LENGTH, batch_size=16,
    )
    np.testing.assert_array_equal(reloaded, label_names[cached_predictions[:len(sample)]])
    print(f"Merged checkpoint reload ({name}): PASS")
    gc.collect()
    torch.cuda.empty_cache()

### Optional local Trackio summary

Trackio records one final, local-only metrics payload after both models pass all comparison gates. It does not request a Hugging Face token, create a Space, or replace the JSON/NumPy/ZIP evidence in Drive. The current training loop does not expose per-step callbacks, so this logs final comparison metrics rather than a live loss curve.

In [ ]:
ENABLE_TRACKIO = True  # @param {type:"boolean"}

trackio_payload = {
    "full_ft/best_dev_macro_f1": full_metrics["best_dev_macro_f1"],
    "full_ft/test_macro_f1": full_metrics["test"]["macro_f1"],
    "full_ft/test_accuracy": full_metrics["test"]["accuracy"],
    "full_ft/training_minutes": full_metrics["training_seconds"] / 60,
    "full_ft/peak_cuda_mb": full_metrics["peak_cuda_memory_mb"],
    "lora/best_dev_macro_f1": lora_metrics["best_dev_macro_f1"],
    "lora/test_macro_f1": lora_metrics["test"]["macro_f1"],
    "lora/test_accuracy": lora_metrics["test"]["accuracy"],
    "lora/training_minutes": lora_metrics["training_seconds"] / 60,
    "lora/peak_cuda_mb": lora_metrics["peak_cuda_memory_mb"],
}
(EXPERIMENT_DIR / "trackio_final_metrics.json").write_text(
    json.dumps(trackio_payload, indent=2), encoding="utf-8"
)

if ENABLE_TRACKIO:
    import trackio

    trackio.init(
        project="hotel-review-nlp",
        name=f"{EXPERIMENT_ID}-{GIT_COMMIT[:8]}",
        config={
            "git_commit": GIT_COMMIT,
            "model": MODEL_NAME,
            "train_rows": EXPECTED_FINGERPRINTS["train"]["rows"],
            "epochs": EPOCHS,
            "full_lr": FULL_LR,
            "lora_lr": LORA_LR,
            "lora_rank": LORA_RANK,
        },
    )
    trackio.log(trackio_payload)
    trackio.finish()
    print("Trackio final metrics: logged locally without HF authentication or Space sync.")
else:
    print("Trackio disabled; the persistent JSON metrics payload was still written.")

## 8. Build full and handoff ZIP files

The full ZIP contains both model directories and all evidence. The much smaller handoff ZIP contains the manifests, configs, metrics, logs, logits, and labels needed to update the repository benchmark and README. Model weights remain in the persistent Drive directory.

In [ ]:
import zipfile

EXPORT_DIR = OUTPUT_ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def file_record(path: Path) -> dict:
    return {
        "path": str(path.relative_to(EXPERIMENT_DIR)).replace("\\", "/"),
        "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }

artifact_records = [
    file_record(path)
    for path in sorted(EXPERIMENT_DIR.rglob("*"))
    if path.is_file() and path.name != "artifact_manifest.json"
]
artifact_manifest = {
    "experiment_id": EXPERIMENT_ID,
    "git_commit": GIT_COMMIT,
    "files": artifact_records,
}
(EXPERIMENT_DIR / "artifact_manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2), encoding="utf-8"
)

full_zip_base = EXPORT_DIR / f"{EXPERIMENT_ID}_full_artifacts"
full_zip_path = Path(shutil.make_archive(str(full_zip_base), "zip", root_dir=EXPERIMENT_DIR))

handoff_names = {
    "environment.json", "data_manifest.json", "artifact_manifest.json",
    "distilbert_comparison.csv", "distilbert_comparison.json",
    "mcnemar_full_vs_lora.json", "metrics.json", "dev_logits.npy",
    "dev_labels.npy", "test_logits.npy", "test_labels.npy", "train.log",
    "run_config.yaml", "request.json", "lora_scratch_config.json",
    "trackio_final_metrics.json",
}
handoff_zip_path = EXPORT_DIR / f"{EXPERIMENT_ID}_handoff.zip"
with zipfile.ZipFile(handoff_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPERIMENT_DIR.rglob("*")):
        if path.is_file() and path.name in handoff_names:
            archive.write(path, path.relative_to(EXPERIMENT_DIR))

for label, path in [("Handoff ZIP", handoff_zip_path), ("Full artifact ZIP", full_zip_path)]:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{label}: {path}")
    print(f"  size={path.stat().st_size / 2**20:.1f} MiB sha256={digest}")

### Download and hand off

The next cell downloads the small evidence bundle. Send that ZIP back with the Colab runtime type used. Keep the full artifact ZIP in Drive; download it too only if you need an offline model backup.

In [ ]:
DOWNLOAD_HANDOFF_ZIP = True  # @param {type:"boolean"}
DOWNLOAD_FULL_ARTIFACT_ZIP = False  # @param {type:"boolean"}

if DOWNLOAD_HANDOFF_ZIP:
    files.download(str(handoff_zip_path))
if DOWNLOAD_FULL_ARTIFACT_ZIP:
    files.download(str(full_zip_path))

print("Send back:", handoff_zip_path.name)
print("The bundle contains both metrics.json files, logits, labels, logs, and manifests.")

## Completion criteria

This experiment is complete only when:

1. both training cells finish without an exception;
2. the exact input fingerprint gate passes;
3. the comparison/McNemar and merged-checkpoint reload gates pass;
4. both ZIP paths and SHA-256 hashes are printed;
5. the handoff ZIP is returned for repository integration.

An interrupted training cell does not constitute a result. Because the current custom loop has no optimizer-resume checkpoints, rerun that model's cell from the beginning after a disconnect.